# Queueing theory - M/M/1: 1 million arrivals

A long run to check whether the simulated averages approach the analytical (steady-state) M/M/1 solution.

The service station is composed of mix-ins, in the same way as objects in OpenTNSim. Because customers are generated by the arrival and service processes (rates, not a list of known customers), only the service station is created as an object.

This notebook provides an example how to set up a queueing simulation. Input that needs to be specified are:
* 'Lambda': the number of arrivals per hour
* 'Mu': the number of departures (services completed) per server per hour
* 'nr_resources': the number of servers of the service station (c in Kendall's notation)

To further characterise the queueing behaviour in the system the user has to specify the probability distributions that characterise the interarrival and service times. This notebook describes an M/M/1 queue (according to the Kendall notation for queueing systems) where both the interarrival times and the service times are assumed to be exponentially distributed.

Currently implemented distributions are:
* 'M': Negative exponential distribution,
* 'Ek': Erlang k distributions (where k needs to be an integer number, e.g. E2, E3 etc).
* 'D': Deterministic lists with known input

NB: the calculation becomes more accurate when a larger number of arrivals (nr_arr) is simulated. Obviously there is a tradeoff between accuracy and calculation time.

NB: 1 million arrivals take roughly half a minute and about 1-1.5 GB of memory.

In [1]:
# OpenQTSim mixins and utils
import openqtsim
from openqtsim.core import Identifiable, HasResource
from openqtsim.core import create_object, inspect_object
import numpy as np
import pandas as pd
from openqtsim.visualizations import plot_gantt

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

In [2]:
example_title = "M/M/1"
Lambda = 8  # nr of arrivals per hour
Mu = 9  # nr of exits (services completed) per server per hour
nr_arr = 1_000_000
arrival_seed = 42  # random stream of the interarrival times
service_seed = 43  # independent random stream of the service times

#### 1. Create service station
Next we create a Server-object (our service station) out of the available mix-ins:

In [3]:
# make your preferred Server class out of available mix-ins.
Server = create_object(
    "Server",
    (
        Identifiable,  # allows to give the object a name and a random ID
        HasResource,   # number of servers (c in Kendall's notation)
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [4]:
inspect_object(Server, show_parameter_table=True)

,parameter,class,default,status
0,name,Identifiable,,mandatory
1,nr_resources,HasResource,1,optional (unused)
2,id,Identifiable,None,optional (unused)


We define the required input and create the server object:

In [5]:
data_server = {
    "name": "Server",     # required by Identifiable
    "nr_resources": 1,    # optional for HasResource: one server
}

server = Server(**data_server)

#### 2. Define arrival and service processes
Customers are generated from negative exponential interarrival and service times:

In [6]:
A = openqtsim.ArrivalProcess("M", arr_rate=Lambda)
S = openqtsim.ServiceProcess("M", srv_rate=Mu)

#### 3. Run simulation
The server object sets the number of servers of the queue.

In [7]:
%%time
q = openqtsim.Queue(A, S, station=server)
print('Start simulation of "{}" queue'.format(q.kendall_notation))

sim = openqtsim.Simulation(q, arrival_seed=arrival_seed, service_seed=service_seed)
sim.run(nr_arr)

Start simulation of "M/M/1/inf/inf/FIFO" queue
CPU times: user 26.1 s, sys: 792 ms, total: 26.9 s
Wall time: 26.7 s


In [8]:
summary = sim.get_stats()
print('Input IAT: {:.4f}'.format(1/Lambda))
print('Input ST: {:.4f}'.format(1/Mu))

Waiting time in units of service time: 8.0496
Observation horizon: 0 to 124932.0798 hours

Rho_system: system utilisation: 0.8900
Rho_server: server utilisation: 0.8900
P_0: probability nobody in the system: 0.1100

L_s: average nr of customers in the system: 8.0544
L_q: average nr of customers in the queue: 7.1644
W_s: average time spent in the system: 1.0063
W_q: average time spent in the queue: 0.8951

IAT: average inter arrival time: 0.1249
ST: average service time: 0.1112

Input IAT: 0.1250
Input ST: 0.1111


**Analytical solution: M_M_1_inf_inf_FIFO**

    arrival rate: 8.0000 per hour
    service rate: 9.0000 per hour
    rho: 0.8889

    P_0: probability that there are 0 customers in the system = 0.1111
    P_1: probability of 1 customers in the system = 0.0988
    P_10: probability of 10 customers in the system = 0.0342

    no_q: probability that there is nobody in the queue (P_0+P_1) = 0.2099
    util_sys: system utilisation rate  (P_(n>=1)) = 0.8889
    util_ser: server utilisation rate (single server busy fraction) = 0.8889
    P_(n>=2): probability of at least one waiting customer = 0.7901

    L_s: the long term average number of customers present in the system = 8.0000
    L_q: the long term average number of customers present in the queue = 7.1111
    W_s: the long term average time spent in the system = 1.0000 hour (= 60.0 minutes)
    W_q: the long term average waiting time in the queue = 0.8889 hour (= 53.3 minutes)

#### 4. Compare with the analytical solution

For an M/M/1 queue with $\rho = \lambda / \mu < 1$ the steady-state solution is $P_n = (1-\rho)\rho^n$, $L_s = \rho/(1-\rho)$, $L_q = \rho^2/(1-\rho)$, $W_s = 1/(\mu-\lambda)$ and $W_q = \rho/(\mu-\lambda)$. The simulated $P_n$ is the fraction of time with $n$ customers in the system (from the system-state log).

In [9]:
rho = Lambda / Mu
analytical = {
    "IAT": 1 / Lambda, "ST": 1 / Mu, "Rho_server": rho,
    "P_0": 1 - rho, "P_1": (1 - rho) * rho, "P_10": (1 - rho) * rho**10,
    "L_s": rho / (1 - rho), "L_q": rho**2 / (1 - rho),
    "W_s": 1 / (Mu - Lambda), "W_q": rho / (Mu - Lambda),
}

# time-weighted probability of n customers in the system
df, df_sys = sim.return_log()
durations = np.diff(df_sys["t"].to_numpy(), append=summary["observation_end"])
time_in_state = pd.Series(durations).groupby(df_sys["c_s"].to_numpy()).sum() / summary["observation_end"]
simulated = {key: summary[key] for key in analytical if key in summary}
simulated.update({"P_1": time_in_state.get(1, 0.0), "P_10": time_in_state.get(10, 0.0)})

comparison = pd.DataFrame({"analytical": analytical, "simulated": simulated}).reindex(list(analytical))
comparison["difference (%)"] = 100 * (comparison["simulated"] - comparison["analytical"]) / comparison["analytical"]
comparison.round(4)

,analytical,simulated,difference (%)
IAT,0.1250,0.1249,-0.0550
ST,0.1111,0.1112,0.0744
Rho_server,0.8889,0.8900,0.1288
P_0,0.1111,0.1100,-1.0304
P_1,0.0988,0.0979,-0.8918
P_10,0.0342,0.0339,-1.0518
L_s,8.0000,8.0544,0.6803
L_q,7.1111,7.1644,0.7493
W_s,1.0000,1.0063,0.6256
W_q,0.8889,0.8951,0.6945


With 1 million arrivals the simulated averages are within about 1% of the analytical solution. Note that at a high utilisation ($\rho \approx 0.89$) successive waiting times are strongly correlated, so the averages converge much more slowly than the sample size suggests: with 20 or 10,000 arrivals the deviations can still be large. Changing the seeds gives an impression of the remaining statistical uncertainty.

In [10]:
sim.plot_system_state()

In [11]:
sim.plot_IAT_ST()

#### 5. Inspect output

In [12]:
df.head(10)

# Meaning of the columns:
# c_id: customer id
# IAT: interarrival time 
# ST: service time 
# AT: arrival time 
# TSB: time service begins
# TSE: time service ends 
# TCSS: time customer spends in the system
# TCWQ: time customer waits int he queue
# ITS: idle time server
# s_id: id of server assigned to customer

,c_id,name,length,maxwait,status,departure,IAT,ST,AT,TSB,TSE,TCSS,TCWQ,ITS,s_id
0,1,None,None,None,served,0.150767,0.121097,0.029670,0.121097,0.121097,0.150767,0.029670,0.000000,0.121097,1
1,2,None,None,None,served,0.568504,0.228537,0.218870,0.349634,0.349634,0.568504,0.218870,0.000000,0.198868,1
2,3,None,None,None,served,0.606197,0.057185,0.037693,0.406819,0.568504,0.606197,0.199378,0.161685,0.000000,1
3,4,None,None,None,served,0.785918,0.034658,0.179721,0.441477,0.606197,0.785918,0.344441,0.164720,0.000000,1
4,5,None,None,None,served,0.909832,0.408205,0.060150,0.849682,0.849682,0.909832,0.060150,0.000000,0.063764,1
5,6,None,None,None,served,0.934007,0.021258,0.024175,0.870940,0.909832,0.934007,0.063066,0.038891,0.000000,1
6,7,None,None,None,served,0.950087,0.006404,0.016080,0.877345,0.934007,0.950087,0.072742,0.056662,0.000000,1
7,8,None,None,None,served,0.982219,0.019225,0.032132,0.896570,0.950087,0.982219,0.085649,0.053517,0.000000,1
8,9,None,None,None,served,1.496717,0.557365,0.042782,1.453935,1.453935,1.496717,0.042782,0.000000,0.471716,1
9,10,None,None,None,served,1.554311,0.020834,0.057594,1.474769,1.496717,1.554311,0.079542,0.021948,0.000000,1


In [13]:
df_sys.head(10)

# Meaning of the columns:
# t: time of logging (if 'lambda' and 'mu' are specified as values per hour, the unit of 't' is hours)
# c_s: nr of customers in the system
# c_q nr of customers in the queue

,t,c_s,c_q
0,0.000000,0,0
1,0.121097,1,1
2,0.121097,1,0
3,0.150767,0,0
4,0.349634,1,1
5,0.349634,1,0
6,0.406819,2,1
7,0.441477,3,2
8,0.568504,2,2
9,0.568504,2,1


#### 6. Gantt chart: first 20 customers

A Gantt chart of 1 million customers is unreadable, so only the first 20 customers are shown. Orange spans arrival (`AT`) to service start (`TSB`); blue spans service start to departure (`TSE`). Hover to inspect the times in hours (and days) since simulation start.

In [14]:
first = df.sort_values(["AT", "c_id"]).head(20)
fig = plot_gantt(first, title=f"{example_title}: first {len(first)} of {len(df):,} customers "
                             f"(arrival seed {arrival_seed}, service seed {service_seed})")
fig.show()
first[["c_id", "AT", "TSB", "TSE", "TCWQ", "ST", "TCSS", "s_id"]]

,c_id,AT,TSB,TSE,TCWQ,ST,TCSS,s_id
0,1,0.121097,0.121097,0.150767,0.000000,0.029670,0.029670,1
1,2,0.349634,0.349634,0.568504,0.000000,0.218870,0.218870,1
2,3,0.406819,0.568504,0.606197,0.161685,0.037693,0.199378,1
3,4,0.441477,0.606197,0.785918,0.164720,0.179721,0.344441,1
4,5,0.849682,0.849682,0.909832,0.000000,0.060150,0.060150,1
5,6,0.870940,0.909832,0.934007,0.038891,0.024175,0.063066,1
6,7,0.877345,0.934007,0.950087,0.056662,0.016080,0.072742,1
7,8,0.896570,0.950087,0.982219,0.053517,0.032132,0.085649,1
8,9,1.453935,1.453935,1.496717,0.000000,0.042782,0.042782,1
9,10,1.474769,1.496717,1.554311,0.021948,0.057594,0.079542,1
